In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
from rag_helper import RAGBase


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [4]:
answer = assistant.rag('How do I run Ollama locally?')
print(answer)

To run Ollama locally:

1. Install Ollama from https://ollama.com/download for your OS:
   - macOS: download and install the `.pkg`
   - Windows: download and install the `.msi`
   - Linux: run:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. Open a terminal and start a local model:
   ```bash
   ollama run llama3
   ```

   This downloads the LLaMA 3 model, starts it locally, and opens a chat-like interface.

3. To test that the local Ollama server is running:
   ```bash
   curl http://localhost:11434
   ```

   You should get a response like:
   ```json
   {"models": [...]}  
   ```

If you need to use Ollama from Python, install the client with:

```bash
pip install ollama
```

and use:

```python
import ollama

response = ollama.chat(
    model='llama3',
    messages=[{"role": "user", "content": your_prompt}]
)

print(response['message']['content'])
```


In [5]:
answer = assistant.rag('How do I run Olama locally?')
print(answer)

I don’t see any FAQ entry about running **Ollama** locally.

The only local-run guidance in the context says you can run the **course locally** instead of Codespaces if you’re comfortable setting up Python, `uv`, Jupyter, Docker, and any other needed tools, and that you should document your setup and keep it reproducible.

If you meant something else by “Olama,” I’d need more context from the FAQ database.


In [6]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
)

response.output_text

'Usually, yes — if the course is still open and you meet any prerequisites.\n\nTo check quickly:\n1. See whether enrollment is still open.\n2. Confirm if there are any requirements or waitlists.\n3. Contact the course instructor, coordinator, or registrar/admissions office if you’re unsure.\n\nIf you want, I can help you draft a short message asking to join the course.'

In [7]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [8]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

In [9]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [10]:
call = response.output[0]

In [11]:
call

ResponseFunctionToolCall(arguments='{"query":"join course discovered can I join course enrollment late join discovered"}', call_id='call_ouedLcAn09ErOc9PSd0UAc4X', name='search', type='function_call', id='fc_0f8ca501119cc0d7006a5eeca15da0819ba61cf2752616ed9c', namespace=None, status='completed')

In [13]:
import json

args = json.loads(call.arguments)
args

{'query': 'join course discovered can I join course enrollment late join discovered'}

In [14]:
results = search(**args)

In [15]:
result_json = json.dumps(results, indent=2)

In [16]:
function_call_output = {
    "type": "function_call_output",
    'call_id': call.call_id,
    'output': result_json,
}

In [17]:
messages.append(call)

In [18]:
result_json

'[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",\n    "answer": "No, you can only get a certificate if you finish the course with a \\"live\\" cohort.\\n\\nTo get the certificate, you need to finish a capstone project and complete the\\nrequired peer reviews. Homework is not required. You can work through the\\nmaterial and prepare your project in self-paced mode, but project submission and\\npeer review must happen while a live cohort is accepting them."\n  },\n  {\n    "id": "bd31146b0e",\n    "

In [19]:
messages.append(function_call_output)

In [20]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [21]:
print(response.output_text)

Yes — you can still join the course and start learning.

If you want a certificate, you’ll need to submit your project while submissions are still open.
